# மனிதர்-இன்-தி-லூப்: முன்னிலை நடைமுறைகள், ஆபத்து மட்டப்பக்கவிளைவுகள் மற்றும் கண்காணிப்பு பதிவு

இந்த பாடத்துக்கான README மனிதர்-இன்-தி-லூப் ஐ ஒரு சிறிய எடுத்துக்காட்டுடன் அறிமுகப்படுத்துகிறது, அதில் முகவர் ஏற்கனவே பதிலை உருவாக்கிய பின் பயனரிடம் `அனுமதி` அல்லது `நிராகரிப்பு` கேட்கப்படுகிறது. அந்த முறை சிறந்த தொடக்கமாகும், ஆனால் உற்பத்தி HITL செயல்பாடுகள் பொதுவாக ஆறு கூடுதல் கூறுகளை தேவைப்படுத்துகின்றன:

1. ஆபத்தான படிக்கடி செயல்படுத்தும் முன் **முன்ன நிலை நடைமுறை** இயங்க வேண்டும், ஆகையால் செலவு, மாற்றமிக்க தன்மை, மற்றும் தாமதம் கட்டுப்பாட்டில் உள்ளன.
2. **ஆபத்து மட்டப்பக்கவிளைவுகள்**, இப்படி குறைந்த ஆபத்துள்ள செயல்கள் தானாக இயங்கிடும், மிதமான ஆபத்துள்ள செயல்கள் தொகுப்பாக அனுமதி பெறும், ஆனால் மிக அதிக ஆபத்துள்ள செயல்கள் மட்டுமே மனிதரைத் தடுக்கின்றன.
3. ஒரு **கண்காணிப்பு பதிவு மற்றும் திருத்தல் சுற்று**, ஆகையால் ஒவ்வொரு நடைமுறை முடிவும் JSONL ஆக பதிவாகி, ஒரு நிராகரிப்பு structured காரணத்துடன் முகவருக்கு மறுபடி கேட்கப்படும், வெறும் `சரிசெய்தல்...` அச்சிடுவதைவிட சிறந்தது.

இந்த நோட்புக் இவை அனைத்தையும் `06-system-message-framework.ipynb` என்ற அதே அடிப்படைகளின் மேலே கட்டினம் செய்கிறது. இது `DEMO_MODE = True` (இணையவழி உள்ளீடு தேவையில்லை) அல்லது உண்மையான `input()` கேட்கையில் செயல்படுகிறது, அதாவது `DEMO_MODE = False`. குறிப்பாக: DEMO_MODE இல் மூன்றாவது குறிக்கோளின் மறுபரிசீலனை ஸ்கிரிப்ட் செய்யப்பட்டுள்ளது அந்த சுற்று முழுவதும் தெளிவாகக் காணப்படுகிறது. உண்மையான மறுபரிசீலனை வழிவழி வகைப்பாடு DEMO_MODE = False மற்றும் ஒரு ஆபரேட்டரைத் தேவைப்படுத்தும்.

**கடந்த பாடங்களுடன் உட்படா (வேறு பாடங்களில் கையாளப்படுகிறது):** அங்கீகாரம் மற்றும் அணுகலை கட்டுப்பாடு (பாடம் 06 README அச்சுறுத்தல் 2), கருவி-கால் மிடில்웨어 (பாடம் 14 MAF ஆழ்ந்த ஆய்வு), பல முகவர் விவாத முறைகள்.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## மாதிரி 1: முன்னொரு-அமல்படுத்தும் கதவு

README இல் உள்ள HITL துண்டு முதலில் முகவரியை அழைக்கிறது, பின்னர் பயனரிடம் வெளியீட்டை அனுமதிக்க கேட்கிறது. இது ஒரு **பின்னொரு-அமல்படுத்தும்** ஓட்டமாகும். முகவர் ஏற்கனவே செயல்படுத்தப்பட்டுருப்பதால், LLM அழைப்பின் செலவு ஏற்கனவே கட்டப்பட்டு இருக்கிறது, மற்றும் எந்த உபபிரவேசமும் (அனுப்பிய மின்னஞ்சல், எழுதப்பட்ட தரவுத்தள வரிசை, பதிந்த கருத்து) ஏற்கனவே நடந்துவிட்டது.

ஒரு **முன்னொரு-அமல்படுத்தும்** ஓட்டம் முகவரி ஆபத்தான படியை இயக்குவதற்கு முன் கதவை இடைக்காலமாக சேர்க்கிறது. முகவர் நடவடிக்கையை முன்மொழிக்கிறது, கதவு செயல்படுத்த வேண்டுமா என்பதைக் தீர்மானிக்கிறது, மற்றும் அனுமதித்தால் மட்டுமே உபபிரவேசம் நிகழ்கிறது.

| அம்சம் | பின்னொரு-அமல்படுத்தல் (README துண்டு) | முன்னொரு-அமல்படுத்தும் கதவு (இந்த நோட்புக்) |
|---|---|---|
| அனுமதி எப்போது செயல்படுகிறது? | முகவர் வெளியீடு தயாரித்தபின் | எந்த உபபிரவேசம் அமுல்படுத்தப்படுவதற்கு முன் |
| மறுப்பில் LLM செலவு | ஏற்கனவே கட்டப்பட்டுள்ளது | செயல்பாட்டிற்கு அல்லாமல் முன்மொழிவிற்கு மட்டுமே செலுத்தப்படுகிறது |
| திரும்ப முடியாத உபபிரவேசங்கள் | சாத்தியமாகும் (செயல் ஏற்கனவே நடைபெற்றது) | தடுக்கப்பட்டது |
| ஆய்வுப் பரிசுத்தம் | அனுமதி ஒரு அச்சுப்பொதி | அனுமதி ஒரு JSON பதிவாகும், நேரம், செயல்பாடு, காரணத்துடன் |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## வடிவமைப்பு 2: ஆபத்து அடுக்குமுறை

ஒவ்வொரு செயலுக்கும் மனித அனுமதி அவசியம் அல்ல. பொது API-க்கு வாசிக்க மட்டும் lookup செய்வது, வாடிக்கையாளர் மின்னஞ்சல் அனுப்புவதைவிட வேறுபட்ட ஆபத்துக்கள் உள்ளன. இரண்டும் ஒன்றாக கையாளுவது இயக்குனரின் கவனத்தை வீணாக்கி, முகவரியின் வேகத்தை குறைக்கும்.

ஒரு எளிய 3-அடுக்கு மாதிரி:

| அடுக்கு | உதாரணங்கள் | அனுமதி ஓட்டம் |
|---|---|---|
| `குறைவு` (வாசிக்க மட்டும்) | அறிவுத்தளம் தேடல், விமான விருப்பங்கள் தேடல், பொது வலைப்பக்கத்தை எடுத்தல் | தானாக செயல்படுத்தல், சரிபார்ப்புக்காக பதிவு |
| `மைய நிலையம்` (சலுகையுடன் மாற்றம்) | முடிவை தொகுப்பு, எண்ணிக்கையை அதிகரிப்பு, முன்னறிவிப்பு திட்டமிடல் | தானாக செயல்படுத்தல், ஆனால் தினசரி மதிப்பீடு |
| `உயர்` (வெளிப்புற முகவரான அல்லது மாற்றமுடியாத) | மின்னஞ்சல் அனுப்பல், கார்டுக்கு கட்டணம் வசூல், பொது சேனலில் இடுகை | மனித அனுமதியில் தடுப்பு |

இது ஒரு அடுக்குமுறை மாதிரி. உற்பத்தி அமைப்புகள் அதிக விவரமான அடுக்குகளைப் பயன்படுத்தும் (எ.கா., AWS IAM அனுமதி நிலைகள், பங்குதாரர் அடிப்படையிலான அணுகல் அடுக்குகள்). கீழுள்ள 3-அடுக்கு வடிவம் வாசிக்க மட்டும் மற்றும் பக்கவிளைவுகள் கொண்ட நடவடிக்கைகளை கலந்தடக்கும் முகவரிக்கு மிகச் சிறிய பயன்பாட்டுக் கூடிய வடிவம்.

கீழுள்ள வகைப்பான், கீ-வர்த்தக நுட்பங்களைப் பயன்படுத்தி, டெமோ தீர்மானமானதும் மலிவானதும் ஆகத் தனக்காக உள்ளது. உற்பத்தியில் நீங்கள் இதனை கற்ற வகைப்பான் அல்லது கொள்கை இயந்திரத்துடன் மாற்றுவீர்கள்.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## மாதிரி 3: கணக்கெடுப்பு பதிவு மற்றும் திருத்தச் சுற்றம்

ஒரு `print("Response approved.")` என்பது கணக்கெடுப்பு பதிவு அல்ல. நம்பிக்கைக்காக, ஒவ்வொரு வாயில்வே மற்றும் நான்கு முடிவும் பின்னர் விசாரிக்க, மீண்டும் செயல்படுத்த, அல்லது ஒரு சம்பவ மதிப்பீட்டுடன் இணைக்கக்கூடிய கட்டமைக்கப்பட்ட நிகழ்வாக பதிவு செய்யப்பட வேண்டும்.

இரண்டு கூறுகள்:

1. **சேர்க்கை மட்டுமுள்ள JSONL.** ஒரு முடிவுக்கு ஒரு வரி, நேரமுத்திரை, செயல்பாடு, நிலை, முடிவு, காரணம் ஆகியவை உள்ளன. தேட எளிது, பின்னர் ஒரு உண்மையான பதிவு களஞ்சியத்திற்கு அனுப்ப எளிது.
2. **நிராகரிப்பில் திருத்தச் சுற்றம்.** வாயில் `deny` என்ற முடிவை தரும்போது, முகவரி மறுக்கல் காரணத்துடன் தனக்கே மீண்டும் கேள்வி கேட்டு, அடுத்த முன்மொழிவில் பிரச்சினையை தவிர்க்க முடியும்.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## கூடுதல் வளங்கள்

பல மற்ற பொது திட்டங்கள் இந்த HITL வடிவமைப்புகளின் வேறுபாடுகளை செயல்படுத்துகின்றன. உங்கள் ஸ்டாக்கிற்கு பொருத்தமான அணுகுமுறைகளை கண்டுபிடிக்க ஒப்பிடுக:

- **LangChain** மனித-இன்-லூப் கருவி மூடரிகளானவை ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): மனித உள்ளீட்டிற்கு இடைவெளி தரும் டிராப்-இன் கருவி மூடரிகள்.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ இதை மறுசீரமைத்தது): பல முகவர் உரையாடல்களில் மனிதனை பிரதிபலிக்க முகவர் கதாபாத்திரமாக பயன்படுத்துகிறது.
- **Microsoft Agent Framework (MAF)** ფუნქ்ஷன்-அழைப்பு மிடில்வேர் ([docs](https://learn.microsoft.com/agent-framework/)): ஒவ்வொரு கருவி/ഫунк്ഷன் அழைப்பிலும் இயங்கும் மிடில்வேர், மதிப்பாய்வு மற்றும் அனுமதி ஒழுங்குகளுக்கு பொருத்தமானது.

ஒவ்வொரு திட்டமும் மூன்று துணை வடிவங்களை வேறுபட்டவாறு கையாள்கிறது: LangChain அவற்றை கருவிகளாக மூடி, AutoGen முகவர்க் கதாபாத்திரம் பயன்படுத்துகிறது, மற்றும் Microsoft Agent Framework ஃபங்க்ஷன்-அழைப்பு மிடில்வேர் பயன்படுத்துகிறது. உங்கள் சொந்த முகவருக்கான வடிவமைப்பைத் தேர்ந்தெடுப்பதற்கு முன் ஒரு அல்லது இரண்டு செயலாக்கங்களை முழுமையாக படியுங்கள்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
